In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install -qq -U evaluate rouge_score

In [2]:
import unsloth
from unsloth import FastLanguageModel
import torch

import os
os.environ["UNSLOTH_RETURN_LOGITS"] = "1"
max_seq_length = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Distill-Qwen-1.5B-bnb-4bit",
    max_seq_length = max_seq_length,   # Define context length
    load_in_4bit = True,     # 4bit uses much less memory
    load_in_8bit = False,    # A bit more accurate, uses 2x memory
    full_finetuning = False, # We have full finetuning now!
    # token = "hf_...",      # Add your token if using a gated model
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-03-24 16:16:13.248116: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774368973.270959     209 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774368973.278648     209 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774368973.297889     209 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774368973.297908     209 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774368973.297911     209 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.11: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/deepseek-r1-distill-qwen-1.5b-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [ ]:
# Add LoRA adapters to the model
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,           # LoRA rank (higher rank = more parameters, potentially better fit but more memory)
    target_modules = [.
                      "q_proj", "k_proj", "v_proj", "o_proj", # Target attention and MLP layers
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,  # Scaling factor (often set to r or 2*r)
    lora_dropout = 0, # Dropout probability for LoRA layers
    bias = "none",    # Fine-tuning bias terms ('none' is often optimal)
    # Use Unsloth's gradient checkpointing for memory saving
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False, # Rank Stable LoRA (optional)
    loftq_config = None, # LoftQ initialization (optional)
)

Unsloth 2026.3.11 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — LOAD + COMBINE DATASETS
# ─────────────────────────────────────────────────────────────────────────────
print("[ 2/5 ] Loading datasets")
from datasets import load_dataset  , concatenate_datasets
import random
import gc
SEED = 42

SEED           = 42
TARGET_TOTAL   = 20000        # 20k is sufficient — quality >> quantity for SLMs

WEIGHTS = {
    "instruction":   0.35,
    "summarization": 0.30,
    "extraction":    0.20,
    "decomposition": 0.15,
}

ds1 = load_dataset("Rohit-Katkar2003/Instruction-dataset",   split="train")
ds2 = load_dataset("Rohit-Katkar2003/summarization-dataset", split="train")
ds3 = load_dataset("Rohit-Katkar2003/Extractional-dataset",  split="train")
ds4 = load_dataset("Rohit-Katkar2003/decomposition-dataset", split="train")

print(f"    D1={len(ds1):,}  D2={len(ds2):,}  D3={len(ds3):,}  D4={len(ds4):,}")

random.seed(SEED)
splits = []
for name, ds, w in [
    ("instruction",   ds1, WEIGHTS["instruction"]),
    ("summarization", ds2, WEIGHTS["summarization"]),
    ("extraction",    ds3, WEIGHTS["extraction"]),
    ("decomposition", ds4, WEIGHTS["decomposition"]),
]:
    target_n = int(TARGET_TOTAL * w)
    if len(ds) >= target_n:
        idx = random.sample(range(len(ds)), target_n)
    else:
        repeats = (target_n // len(ds)) + 1
        idx     = (list(range(len(ds))) * repeats)[:target_n]
        random.shuffle(idx)
        print(f"    Upsampled {name}: {len(ds)} → {target_n}")
    splits.append(ds.select(idx))

combined = concatenate_datasets(splits).shuffle(seed=SEED)
del ds1, ds2, ds3, ds4
gc.collect()
print(f"[ 2/5 ] Combined: {len(combined):,} samples ✓")

[ 2/5 ] Loading datasets


Extraction.jsonl:   0%|          | 0.00/86.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

decomposition.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/500 [00:00<?, ? examples/s]

    D1=71,179  D2=100,000  D3=100,000  D4=500
    Upsampled decomposition: 500 → 3000
[ 2/5 ] Combined: 20,000 samples ✓


In [6]:
combined[1]

{'dataset': 'squad_v2',
 'task': 'information_extraction',
 'system': 'You are an information extraction agent. Extract precise facts from text. Return valid JSON. If the answer cannot be found, return null for that field.',
 'messages': [{'role': 'user',
   'content': "Analyze this text and extract the requested fact. If not present, state null.\n\nText:\nIn the 1970s, job losses due to industrial restructuring caused New York City to suffer from economic problems and rising crime rates. While a resurgence in the financial industry greatly improved the city's economic health in the 1980s, New York's crime rate continued to increase through that decade and into the beginning of the 1990s. By the mid 1990s, crime rates started to drop dramatically due to revised police strategies, improving economic opportunities, gentrification, and new residents, both American transplants and new immigrants from Asia and Latin America. Important new sectors, such as Silicon Alley, emerged in the city'

In [7]:
def format_chat(example):
    system = example["system"]
    messages = example["messages"]

    text = f"<|system|>\n{system}\n"

    for msg in messages:
        if msg["role"] == "user":
            text += f"<|user|>\n{msg['content']}\n"
        elif msg["role"] == "assistant":
            text += f"<|assistant|>\n{msg['content']}\n"

    return {"text": text}

train_dataset = combined.map(format_chat)

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

In [8]:
split_dataset = train_dataset.train_test_split(test_size=0.15, seed=3407)
# Access the train and validation sets
train_dataset = split_dataset["train"]
valid_dataset = split_dataset["test"]

print("Train size:", len(train_dataset))
print("Valid size:", len(valid_dataset))
print(train_dataset[0])

Train size: 17000
Valid size: 3000
{'dataset': 'no_robot', 'task': 'instruction_tune', 'system': 'You are Helpful assistant.', 'messages': [{'role': 'user', 'content': 'What is the biggest continent?'}, {'role': 'assistant', 'content': "There are various ways to measure and compare continents. Asia is the biggest continent on Earth in terms of land mass. Asia's area is 17.2 million square miles. Asia also has the largest population among the seven continents, with over 4.5 billion people. The next largest continent in terms of land mass is Africa. Africa is 11.6 million square miles in area, and is home to 1.3 billion people."}], 'text': "<|system|>\nYou are Helpful assistant.\n<|user|>\nWhat is the biggest continent?\n<|assistant|>\nThere are various ways to measure and compare continents. Asia is the biggest continent on Earth in terms of land mass. Asia's area is 17.2 million square miles. Asia also has the largest population among the seven continents, with over 4.5 billion people.

In [9]:
print("<|assistant|>" in train_dataset[0]["text"])

True


In [10]:
import numpy as np
from transformers import EvalPrediction
import evaluate

# Load metrics
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

In [11]:
def preprocess_logits_for_metrics(logits, labels):
    """Returns predicted token IDs (argmax) for metrics calculation"""
    if isinstance(logits, tuple):
        logits = logits[0]  # Unpack if needed
    return logits.argmax(dim=-1)

def compute_metrics(eval_preds: EvalPrediction):
    """Compute ROUGE, BLEU, AND token-level accuracy"""
    preds, labels = eval_preds

    # --- Token-Level Accuracy Calculation ---
    # Flatten all predictions/labels (ignore padding tokens)
    mask = labels != -100  # Only compare non-ignored tokens
    preds_flat = preds[mask].flatten()
    labels_flat = labels[mask].flatten()

    accuracy = (preds_flat == labels_flat).mean()

    # --- Text Generation Metrics ---
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Post-process text
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]

    rouge_results = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )
    bleu_results = bleu.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    return {
        "accuracy": float(accuracy),  # Token-level exact match
        "rouge1": rouge_results["rouge1"],
        "rouge2": rouge_results["rouge2"],
        "rougeL": rouge_results["rougeL"],
        "bleu": bleu_results["bleu"],
    }

def compute_metricsR(eval_preds: EvalPrediction):
    """Compute ROUGE, BLEU, AND token-level accuracy"""
    preds, labels = eval_preds

    # --- Token-Level Accuracy Calculation ---
    # Flatten all predictions/labels (ignore padding tokens)
    mask = labels != -100  # Only compare non-ignored tokens
    preds_flat = preds[mask].flatten()
    labels_flat = labels[mask].flatten()

    accuracy = (preds_flat == labels_flat).mean()

    # --- Text Generation Metrics ---
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Post-process text
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]

    rouge_results = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )
    bleu_results = bleu.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    return {
        "accuracy": float(accuracy),  # Token-level exact match
        "rouge1": rouge_results["rouge1"],
        "rouge2": rouge_results["rouge2"],
        "rougeL": rouge_results["rougeL"],
        "bleu": bleu_results["bleu"],
    }

In [12]:
type(train_dataset)

datasets.arrow_dataset.Dataset

In [16]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=None,

    args=SFTConfig(
        dataset_text_field="text",

        # 🔥 BATCH (IMPORTANT)
        per_device_train_batch_size=4,
        gradient_accumulation_steps=8,   # effective batch = 32

        # 🔥 TRAINING LENGTH
        num_train_epochs=1,
        # max_steps=100,                    # REMOVE 100 step limit

        # 🔥 LEARNING RATE (VERY IMPORTANT)
        learning_rate=2e-4,              # best for LoRA
        warmup_ratio=0.05,               # better than fixed steps

        # 🔥 OPTIMIZATION
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",      # better than linear

        # 🔥 STABILITY
        max_grad_norm=1.0,

        # 🔥 PERFORMANCE
        logging_steps=10,
        dataloader_num_workers=2,
        dataloader_pin_memory=True,

        # 🔥 PRECISION
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),

        # 🔥 MISC
        seed=3407,
        report_to="none",
    ),

    # 🔥 HUGE IMPACT (DON'T SKIP)
    packing=True,
)

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [17]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
4.291 GB of memory reserved.


In [18]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 17,000 | Num Epochs = 1 | Total steps = 532
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 8 x 1) = 32
 "-____-"     Trainable parameters = 18,464,768 of 1,795,552,768 (1.03% trained)


Step,Training Loss
10,2.129300
20,1.964300
30,1.957000
40,1.996300
50,1.960400
60,1.903800
70,1.953800
80,1.899100
90,1.885400
100,2.116700


In [24]:
from huggingface_hub import login  
import os  
HF_WRITE_TOKEN = os.getenv("HUGGINFACE_WRITE_TOKEN")
login(HF_WRITE_TOKEN) 

#  IMPORTANT (only if using LoRA / PEFT)
# model = model.merge_and_unload()

# Save model + tokenizer
model.save_pretrained("deepseek_r1_finetune")
tokenizer.save_pretrained("deepseek_r1_finetune")

model.push_to_hub("Rohit-Katkar2003/deepseek_r1_finetune")
tokenizer.push_to_hub("Rohit-Katkar2003/deepseek_r1_finetune") 


README.md:   0%|          | 0.00/590 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/Rohit-Katkar2003/deepseek_r1_finetune


README.md:   0%|          | 0.00/596 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

In [28]:
model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536, padding_idx=151665)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=1536, out_features=1536, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear4bit(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((153

In [30]:
!df -h

Filesystem        Size  Used Avail Use% Mounted on
overlay           7.9T  6.7T  1.2T  85% /
tmpfs              64M     0   64M   0% /dev
shm                14G  8.0K   14G   1% /dev/shm
/dev/loop1         20G  3.1G   17G  16% /kaggle/lib
/dev/sda1         122G   19G  104G  16% /opt/bin
/dev/mapper/snap  7.9T  6.7T  1.2T  85% /etc/hosts
tmpfs              16G     0   16G   0% /proc/acpi
tmpfs              16G     0   16G   0% /proc/scsi
tmpfs              16G     0   16G   0% /sys/firmware


In [39]:
from unsloth import FastLanguageModel
from transformers import TextStreamer
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Rohit-Katkar2003/deepseek_r1_finetune",
    max_seq_length=1024,
    load_in_4bit=True
)

FastLanguageModel.for_inference(model)


##

==((====))==  Unsloth 2026.3.11: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536, padding_idx=151665)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=1536, out_features=1536, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear4bit(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((153

In [42]:
messages = [
    {"role": "system", "content": "You are a helpful AI assistant. Give short and clear answers."},
    {"role": "user", "content": "Explain machine learning simply."}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to("cuda")

streamer = TextStreamer(tokenizer, skip_prompt=True)

outputs = model.generate(
    **inputs,
    streamer=streamer,
    max_new_tokens=512,
    temperature=0.3,
    top_p=0.85,
    repetition_penalty=1.1,
    no_repeat_ngram_size=3
)

Okay, so I need to explain machine learning in simple terms. Let me think about how to approach this.

First, I should define what machine learning is. It's a type of AI that allows systems to learn from data without being explicitly programmed. That makes sense because it's about machines improving through experience.

I should mention the key components: data, algorithms, models. Data is the information the system uses, algorithms are the methods or rules the system follows, and models are the structures that represent the learned patterns.

Wait, maybe I can break it down further. How does it work? Well, when you give a machine learning model some data, like images or text, the algorithm finds patterns on its own. Then, based on those patterns, it makes predictions or decisions without being told exactly what to do.

It's important to note that ML doesn't require programming. Instead, it lets the data speak for itself. So, even if I don't know much about ML, just by giving it data, 

#########

ERROR:hf-to-gguf:Error: /deepseek_r1_finetune is not a directory
